# Дообучение RuBERT-base-cased для классификации спам-сообщений

Ноутбук выполняет дообучение модели `DeepPavlov/rubert-base-cased` (180M параметров, полноразмерный RuBERT) на собранном датасете.

Особенности обучения:
- **FP16** — смешанная точность для ускорения обучения на GPU
- **FocalLoss** — функция потерь, фокусирующаяся на сложных примерах
- **EarlyStopping** — ранняя остановка при отсутствии улучшения F1

Обучение выполняется на трёх вариантах текста:
- исходный текст (raw)
- нормализованный текст (normalized)
- предобработанный текст (preprocessed)

Фильтрация: остаются только сообщения, содержащие преимущественно кириллицу.

## Импорт необходимых библиотек

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.transformer import (
    FocalLossTrainer,
    tokenize_function,
    compute_metrics,
    is_mostly_cyrillic,
    get_training_args,
    get_model_config,
    prepare_text_variants,
    benchmark_cpu_inference,
)
from src.config import PROCESSED_DIR, MODELS_DIR

## Чтение обработанного датасета

Загружается `preprocessed.csv` из `data/processed/`.

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'preprocessed.csv', index_col=0)

## Подготовка вариантов текста

Из датасета выделяются три варианта текста для дообучения:
- **raw** — исходный текст (колонка `text`)
- **normalized** — Unicode-нормализация и удаление HTML-тегов
- **preprocessed** — полная предобработка (колонка `text_preprocessed`)

Фильтрация по кириллице: остаются только сообщения, где доля кириллических символов превышает 30%.

In [3]:
variants = prepare_text_variants(df)

ru_variants = {}
for name, vdf in variants.items():
    mask = vdf['text'].apply(lambda t: is_mostly_cyrillic(str(t)))
    ru_variants[name] = vdf[mask].reset_index(drop=True)
    print(f'{name}: {len(ru_variants[name])} сообщений')

raw: 72342 сообщений
normalized: 72355 сообщений
preprocessed: 72503 сообщений


## Дообучение RuBERT-base-cased

Модель: `DeepPavlov/rubert-base-cased`.
Количество классов: 2 (ham=0, spam=1).

### Загрузка модели и токенайзера

In [4]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
model_config = get_model_config(MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Подготовка датасетов и токенизация

Для каждого варианта текста создаётся отдельный train/test сплит (80/20) и токенизируется.

In [5]:
tokenized = {}
for name, vdf in ru_variants.items():
    ds = Dataset.from_pandas(vdf)
    split = ds.train_test_split(test_size=0.2, seed=42)
    train_ds = split['train']
    test_ds = split['test']

    train_tok = train_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )
    test_tok = test_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )

    tokenized[name] = {'train': train_tok, 'test': test_tok}
    print(f'{name}: train={{len(train_tok)}}, test={{len(test_tok)}}')

Map:   0%|          | 0/57873 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Map:   0%|          | 0/14469 [00:00<?, ? examples/s]

raw: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/57884 [00:00<?, ? examples/s]

Map:   0%|          | 0/14471 [00:00<?, ? examples/s]

normalized: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/58002 [00:00<?, ? examples/s]

Map:   0%|          | 0/14501 [00:00<?, ? examples/s]

preprocessed: train={len(train_tok)}, test={len(test_tok)}


### Создание trainer'ов

Используется `FocalLossTrainer`.
Для каждого варианта текста создаётся отдельный trainer.

In [6]:
model_cfg = {
    'learning_rate': 2e-5,
    'epochs': 5,
    'batch_size': 16,
    'fp16': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
}

trainers = {}
for name in tokenized:
    output_dir = f'finetuned_rubert_base_{name}'

    training_args = get_training_args(
        output_dir=output_dir,
        learning_rate=model_cfg['learning_rate'],
        num_train_epochs=model_cfg['epochs'],
        per_device_train_batch_size=model_cfg['batch_size'],
        per_device_eval_batch_size=model_cfg['batch_size'],
        max_length=model_config['max_length'],
        fp16=model_cfg['fp16'],
        gradient_checkpointing=model_config['gradient_checkpointing'],
    )

    trainer = FocalLossTrainer(
        focal_alpha=model_cfg['focal_alpha'],
        focal_gamma=model_cfg['focal_gamma'],
        model=model,
        args=training_args,
        train_dataset=tokenized[name]['train'],
        eval_dataset=tokenized[name]['test'],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainers[name] = trainer

### Запуск дообучения

Обучение выполняется на GPU (если доступно).

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
model.to(device)

Устройство: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

#### На исходных текстах (raw)

In [8]:
trainers['raw'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001615,0.995369,0.988260,0.995763,0.980870
2,No log,0.001367,0.994540,0.986099,0.997863,0.974609
3,No log,0.001649,0.995922,0.989716,0.991964,0.987478
4,No log,0.002751,0.995646,0.988949,0.997523,0.980522
5,No log,0.002511,0.996130,0.990200,0.996478,0.984000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18090, training_loss=0.0012465918242041745, metrics={'train_runtime': 830.6091, 'train_samples_per_second': 348.377, 'train_steps_per_second': 21.779, 'total_flos': 3.80675652671232e+16, 'train_loss': 0.0012465918242041745, 'epoch': 5.0})

In [9]:
trainers['raw'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.002511,5,0.996130,0.990200,0.996478,0.984000


{'eval_loss': 0.002510633086785674,
 'eval_accuracy': 0.996129656507015,
 'eval_f1': 0.9901995099754988,
 'eval_precision': 0.9964776329693554,
 'eval_recall': 0.984}

Сохранение модели.

In [10]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_base_raw')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_base_raw')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_base_raw/tokenizer_config.json',
 '/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_base_raw/tokenizer.json')

#### На нормализованных текстах (normalized)

In [11]:
trainers['normalized'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.000806,0.997443,0.993420,0.990777,0.996077
2,No log,0.000692,0.997858,0.994469,0.995002,0.993937
3,No log,0.001921,0.996476,0.990930,0.988294,0.993581
4,No log,0.001854,0.997167,0.992654,0.997479,0.987874


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14472, training_loss=0.00043120678779893053, metrics={'train_runtime': 664.618, 'train_samples_per_second': 435.468, 'train_steps_per_second': 27.219, 'total_flos': 3.045984065691648e+16, 'train_loss': 0.00043120678779893053, 'epoch': 4.0})

In [12]:
trainers['normalized'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.000692,4,0.997858,0.994469,0.995002,0.993937


{'eval_loss': 0.0006923809996806085,
 'eval_accuracy': 0.9978577845345864,
 'eval_f1': 0.9944692239072257,
 'eval_precision': 0.9950017850767583,
 'eval_recall': 0.9939372325249644}

Сохранение модели.

In [13]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_base_norm')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_base_norm')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_base_norm/tokenizer_config.json',
 '/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_base_norm/tokenizer.json')

#### На предобработанных текстах (preprocessed)

In [14]:
trainers['preprocessed'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001692,0.996759,0.991677,0.998573,0.984875
2,No log,0.001065,0.996621,0.991347,0.995390,0.987337
3,No log,0.000927,0.996759,0.991683,0.997863,0.985579
4,No log,0.001153,0.997311,0.993108,0.997869,0.988393
5,No log,0.001272,0.997035,0.992396,0.997866,0.986986


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18130, training_loss=0.0005519370906470036, metrics={'train_runtime': 833.5963, 'train_samples_per_second': 347.902, 'train_steps_per_second': 21.749, 'total_flos': 3.81524185824768e+16, 'train_loss': 0.0005519370906470036, 'epoch': 5.0})

In [15]:
trainers['preprocessed'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001153,5,0.997311,0.993108,0.997869,0.988393


{'eval_loss': 0.0011527419555932283,
 'eval_accuracy': 0.9973105303082546,
 'eval_f1': 0.9931083230252695,
 'eval_precision': 0.9978693181818182,
 'eval_recall': 0.9883925430882871}

Сохранение модели.

In [16]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_base_p')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_base_p')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_base_p/tokenizer_config.json',
 '/workspace/STANKIN_AntiSpam_Bot/models/finetuned_rubert_base_p/tokenizer.json')

## Замер CPU-инференса

Замеряется latency инференса на CPU для оценки пригодности модели к развёртыванию на сервере без GPU.

In [17]:
test_messages = [
    "Это честное сообщение от пользователя.",
    "🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎",
    "Зарабатывай миллионы **онлайн** прямо сейчас!",
    "Работа на дому, легкий доход. Пиши в личку!",
    "Привет! Как дела? У меня всё отлично.",
    "steam gift 50$ - steamcommunity.com/gift-card/pay/50\n@everyone",
    "Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!",
    "Как же надоели эти сообщения про казино",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: stankin.ru",
    "3-4 часа и 8 тысяч твои!  Пиши  https://t.me/rasmuswork1",
]

sample_texts = test_messages * 10
suffix_map = {'raw': 'raw', 'normalized': 'norm', 'preprocessed': 'p'}
cpu_results = {}
for name, suffix in suffix_map.items():
    model_path = str(MODELS_DIR / f'finetuned_rubert_base_{suffix}')
    m = AutoModelForSequenceClassification.from_pretrained(model_path)
    tok = AutoTokenizer.from_pretrained(model_path)
    cpu_results[name] = benchmark_cpu_inference(m, tok, sample_texts, max_length=model_config['max_length'])
    print(f'{name}: {cpu_results[name]}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

raw: {'avg_ms_per_msg': 7.62, 'p95_ms_per_msg': 10.81, 'throughput_msgs_per_sec': 131.21}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

normalized: {'avg_ms_per_msg': 8.81, 'p95_ms_per_msg': 16.9, 'throughput_msgs_per_sec': 113.49}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

preprocessed: {'avg_ms_per_msg': 6.96, 'p95_ms_per_msg': 8.02, 'throughput_msgs_per_sec': 143.67}
